## Creating a Spark Session with necessary configurations 

In [ ]:
# Creating a Spark Session with mysql and kafka jars from maven
from pyspark.sql import SparkSession

spark = (
    SparkSession 
    .builder 
    .appName("Producing events to kafka topic, streaming and writing data as parquet files and sinking to mysql table") 
    .config("spark.streaming.stopGracefullyOnShutdown", True) 
    .config('spark.jars.packages', 
            'org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0,'  # Kafka integration
            'mysql:mysql-connector-java:8.0.32') # mysql jdbc
    .config("spark.sql.shuffle.partitions", 8)
    .master("local[*]") 
    .getOrCreate()
)

variable declarations

In [ ]:
# specifying the static variables
topic_name = "<the-topic-name>" # i used player-data
bootstrap_server = "kafka:29092" 
table_name = "<table-name-in-mysql>" # i used players_raw_data
database_name = "<database-name-in-mysql>" # i used players_db
user = "<mysql-username>" # i used root
password = "<mysql-login-password>" # i used root
hostname = "<hostname-either-localhost-OR-your-network's-IP-address>" # i used 192.168.1.214 as it was networks IPv4 address
port_number = "<port-number-at-which-mysql-is-running>" # i used 3307, as i had already a running instance of mysql on port number 3306

reading the stream from the topic

In [ ]:
input_df = (
    spark
    .readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_server)
    .option("subscribe", topic_name)
    .option("startingOffsets", "earliest")
    .load()
)

let's see the schema of the input data from the topic

In [ ]:
input_df.printSchema()

based on the schema, value column represents the data from the topic, but at the moment its in binary form, so we have to convert it into a string

In [ ]:
from pyspark.sql.functions import expr

kafka_json_df = input_df.withColumn("value", expr("CAST(value AS STRING)"))

after converting, now we can view the data properly in the value column, let's verify by the schema

In [ ]:
kafka_json_df.printSchema()

defining the input schema from the kafka topic, this is important for streaming

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

# Defining the JSON Schema in PySpark's StructType format
json_schema = StructType([
    StructField("gameID", StringType(), True),
    StructField("info", StructType([
        StructField("players", ArrayType(StructType([
            StructField("ID", StringType(), True),
            StructField("Name", StringType(), True)
        ])), True),
        StructField("recorded_at", StringType(), True)  # StringType for date-time (or you can use TimestampType)
    ]), True)
])

just keeping the value column, extracting all the columns in the value column

In [ ]:
from pyspark.sql.functions import from_json, col

streaming_df = kafka_json_df.withColumn("values_json", from_json(col("value"), json_schema)).selectExpr("values_json.*")

now seeing the schema of the dataframe

In [ ]:
streaming_df.printSchema()

as our data is nested, lets explode the dataframe

In [ ]:
from pyspark.sql.functions import explode

streaming_df = streaming_df.withColumn("player_data", explode("info.players"))

let's define the final dataframe with unnecessary (nested) columns removed and keeping the exploded columns

In [ ]:
final_df = (
    streaming_df
    .withColumn("player_ID", col("player_data.ID"))
    .withColumn("player_name", col("player_data.Name"))
    .drop("info")
    .drop("player_data")
)

finalize the dataframe's schema

In [ ]:
final_df.printSchema()

the function that will be used for 
1. storing the data as parquet files in the output folder
2. storing the data into mysql table
also maintaining the checkpoint folder

In [ ]:
# Python function to write to multiple sinks 
def send_data(df, batch_id):
    print("Batch id: "+ str(batch_id))
    
    # Writing to parquet files
    df.write.format("parquet").mode("append").save("output/player_data.parquet/")
       
    # Writing to mysql table
    mysql_url = f"jdbc:mysql://{hostname}:{port_number}/{database_name}"
    mysql_properties = {
        "user": user,
        "password": password,
        "driver": "com.mysql.cj.jdbc.Driver"
    }
    
    df.write.jdbc(url=mysql_url, table=table_name, mode="append", properties=mysql_properties)
    
    df.show()

Starting the stream for writing the data

In [ ]:
(final_df
 .writeStream
 .foreachBatch(send_data)
 .trigger(processingTime='10 seconds')
 .option("checkpointLocation", "checkpoint_folder")
 .start()
 .awaitTermination())

once done we can stop the kernel and end the spark session

In [ ]:
spark.stop()